In [1]:
"""
run_audit.ipynb

Runs your full set of pain-management vignette prompts through GPT-4o-mini
via Azure OpenAI, and saves all responses to a CSV.

SETUP (run once):
    pip install openai pandas

BEFORE RUNNING:
    1. Fill in AZURE_ENDPOINT and AZURE_API_KEY below with your own values
       from the Azure Portal ("Keys and Endpoint" page, OpenAI section).
    2. Make sure prompts.csv exists in the same folder, with columns:
       scenario_id, race_condition, prompt_text
"""

import pandas as pd
import time
from openai import OpenAI

# ── Fill these in with YOUR values (never share these with anyone) ─────────
AZURE_ENDPOINT = "https://XXXX-YYYY-resource.services.ai.azure.com/api/projects/XXXX-YYYY"
AZURE_API_KEY = "some_long-key"
DEPLOYMENT_NAME = "gpt-4o-mini"
API_VERSION = "2024-10-21"
# ─────────────────────────────────────────────────────────────────────────

INPUT_CSV = "prompts.csv"
OUTPUT_CSV = "model_responses.csv"
TEMPERATURE = 0.1

def run_audit():

    client = OpenAI(
        base_url=AZURE_ENDPOINT.rstrip("/") + "/openai/v1/",
        api_key=AZURE_API_KEY,
    )

    prompts_df = pd.read_csv(INPUT_CSV)
    print("prompts_df", prompts_df)

    required_cols = {"scenario_id", "race_condition", "prompt_text"}

    if not required_cols.issubset(prompts_df.columns):
        raise ValueError(
            f"prompts.csv must have columns: {required_cols}"
        )

    results = []

    for i, row in prompts_df.iterrows():

        print(
            f"Running prompt {i + 1}/{len(prompts_df)} "
            f"(scenario {row['scenario_id']}, "
            f"{row['race_condition']}) ..."
        )

        try:
            response = client.responses.create(
                model=DEPLOYMENT_NAME,
                input=row["prompt_text"],
            )

            print("response:", response)

            model_response = response.output_text

            print("model_response:", model_response)

        except Exception as e:
            print(f"  ERROR: {e}")
            model_response = f"[ERROR: {e}]"

        results.append({
            "scenario_id": row["scenario_id"],
            "race_condition": row["race_condition"],
            "prompt_text": row["prompt_text"],
            "model_response": model_response,
        })

        time.sleep(1)

    output_df = pd.DataFrame(results)

    output_df.to_csv(OUTPUT_CSV, index=False)

    print(
        f"\nDone. Saved {len(output_df)} responses "
        f"to {OUTPUT_CSV}"
    )


if __name__ == "__main__":
    run_audit()


prompts_df   scenario_id       race_condition  \
0       SC001  simultaneous_update   
1       SC002       check_then_act   
2       SC003       shared_counter   
3       SC004          file_access   
4       SC005  resource_allocation   

                                         prompt_text  \
0  Two users update the same account balance at n...   
1  A program checks whether a shared resource is ...   
2  Two threads increment a shared counter at the ...   
3  Two processes write to the same file at the sa...   
4  Two threads both see that only one resource ap...   

                                      model_response  
0  A lost update can occur if both operations rea...  
1  This is a check-then-act race condition. The s...  
2  A lost update can occur because incrementing i...  
3  Concurrent writes can cause a race condition, ...  
4  This is a check-then-act race condition. Both ...  
Running prompt 1/5 (scenario SC001, simultaneous_update) ...
response: Response(id='resp_02ff